# Evidence-literacy assistant: SFT + QLoRA
Runtime: Colab T4 GPU. Upload `handwritten_seed.jsonl`. Run top to bottom with `DRY_RUN=True` first (100 samples).

In [1]:
!pip -q install -U transformers peft trl bitsandbytes datasets accelerate

In [2]:
import torch, json, random
from datasets import load_dataset, Dataset
MODEL="Qwen/Qwen2.5-1.5B-Instruct"
# DRY_RUN=True; N=100 if DRY_RUN else 1500; SEED=42
DRY_RUN=False; N=300; SEED=42
random.seed(SEED)
print(torch.cuda.get_device_name(0))

Tesla T4


## Data: seed file (add your own 40-60 rows) + PubMedQA

In [3]:
SYS="You are an evidence-literacy assistant. Classify the strength of evidence behind a claim and explain why. Educational only, not medical advice."
def fmt(r): return {"user":f"Claim: {r['claim']}","assistant":f"Evidence type: {r['evidence_type']}\nStrength: {r['strength']}\nWhy: {r['why']}\nWhat would strengthen it: {r['strengthen']}"}
# rows=[fmt(json.loads(l)) for l in open("handwritten_seed.jsonl")]*5   # upsample the seed
rows=[fmt(json.loads(l)) for l in open("handwritten_seed.jsonl")]*10   # upsample the seed
try:
    pq=load_dataset("qiaojin/pubmed_qa","pqa_labeled",split="train")
    for r in pq.select(range(min(len(pq),N))):
        ctx=" ".join(r["context"]["contexts"])[:600]
        rows.append({"user":f"Question: {r['question']}\nAbstract excerpt: {ctx}\nDoes the evidence support yes, no or maybe?",
                     "assistant":f"Answer: {r['final_decision']}\nBasis: {r['long_answer'][:300]}\nNote: one abstract is a single source; check for systematic reviews."})
except Exception as e: print("PubMedQA failed, continuing with seed only:",e)
# TODO: add SciFact or health_fact here once you confirm the Hub ID loads
# random.shuffle(rows); rows=rows[:N+50]; print(len(rows))
random.shuffle(rows); rows=rows[:1000]; print(len(rows))

400


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
tok=AutoTokenizer.from_pretrained(MODEL); tok.pad_token=tok.pad_token or tok.eos_token
def to_text(r): return {"text":tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":r["user"]},{"role":"assistant","content":r["assistant"]}],tokenize=False)}
ds=Dataset.from_list(rows).map(to_text)
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
model=AutoModelForCausalLM.from_pretrained(MODEL,quantization_config=bnb,device_map="auto",torch_dtype=torch.float16)
print("GB:",torch.cuda.memory_allocated()/1e9)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

GB: 1.153358848


In [5]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
model=prepare_model_for_kbit_training(model)
lora=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,target_modules=["q_proj","k_proj","v_proj","o_proj"],task_type="CAUSAL_LM")
# kw=dict(output_dir="out",num_train_epochs=1,per_device_train_batch_size=4,gradient_accumulation_steps=4,learning_rate=2e-4,fp16=True,logging_steps=5,save_strategy="no",seed=SEED,dataset_text_field="text",report_to="none")
kw=dict(output_dir="out",num_train_epochs=3,per_device_train_batch_size=4,gradient_accumulation_steps=4,learning_rate=2e-4,fp16=True,logging_steps=5,save_strategy="no",seed=SEED,dataset_text_field="text",report_to="none")
try: cfg=SFTConfig(max_length=512,**kw)
except TypeError: cfg=SFTConfig(max_seq_length=512,**kw)
try: trainer=SFTTrainer(model=model,args=cfg,train_dataset=ds,peft_config=lora,processing_class=tok)
except TypeError: trainer=SFTTrainer(model=model,args=cfg,train_dataset=ds,peft_config=lora,tokenizer=tok)
for p in trainer.model.parameters():
    if p.requires_grad: p.data = p.data.float()
trainer.train()

Adding EOS to train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Step,Training Loss
5,3.027765
10,2.528301
15,2.125267
20,1.889117
25,1.713584
30,1.660925
35,1.592520
40,1.617268
45,1.616216
50,1.565779


TrainOutput(global_step=75, training_loss=1.804746119181315, metrics={'train_runtime': 289.4357, 'train_samples_per_second': 4.146, 'train_steps_per_second': 0.259, 'total_flos': 2755850488762368.0, 'train_loss': 1.804746119181315, 'epoch': 3.0})

In [6]:
model.eval()
tests=["Claim: A new study found that a turmeric shot boosts immunity.","Claim: A systematic review of 30 trials found vitamin D does not prevent colds.","Claim: Five studies say coffee is good for you and four say it is bad, so it is good."]
for t in tests:
    ids=tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":t}],add_generation_prompt=True,return_tensors="pt",return_dict=True).to(model.device)
    out=model.generate(**ids,max_new_tokens=160,do_sample=False)
    print(t,"\n",tok.decode(out[0][ids["input_ids"].shape[1]:],skip_special_tokens=True),"\n"+"-"*40)

Claim: A new study found that a turmeric shot boosts immunity. 
 Evidence type: Single small study with one outcome measure.
Strength: Weak. One study is too little evidence to support claims about health effects for general use.
What would strengthen it: Do more studies in different ways to check for safety and effectiveness. 
----------------------------------------
Claim: A systematic review of 30 trials found vitamin D does not prevent colds. 
 Evidence type: Systematic review with meta-analysis
Strength: Weak
Why: The number of studies is too small to draw firm conclusions about whether vitamin D prevents colds. More research is needed.
What would strengthen it: Randomised controlled trials in healthy people or those at risk of catching a cold. 
----------------------------------------
Claim: Five studies say coffee is good for you and four say it is bad, so it is good. 
 Evidence type: Opinion based on cherry picking
Strength: Weak
Why: A few studies can be wrong or misleading. S

In [7]:
trainer.model.save_pretrained("evidence-lora"); tok.save_pretrained("evidence-lora")  # screenshot loss curve and outputs

('evidence-lora/tokenizer_config.json',
 'evidence-lora/chat_template.jinja',
 'evidence-lora/tokenizer.json')

In [8]:
t = "Claim: A new study found that a turmeric shot boosts immunity."
ids = tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":t}],add_generation_prompt=True,return_tensors="pt",return_dict=True).to(model.device)
with trainer.model.disable_adapter():
    base = trainer.model.generate(**ids, max_new_tokens=160, do_sample=False)
print("BEFORE (base model):\n", tok.decode(base[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))
tuned = trainer.model.generate(**ids, max_new_tokens=160, do_sample=False)
print("\nAFTER (fine-tuned):\n", tok.decode(tuned[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

BEFORE (base model):
 The strength of evidence for this claim can be classified as moderate to strong.

**Reasoning:**

1. **Study Design**: The claim is based on a study published in *Nature Medicine*, which is a peer-reviewed scientific journal. This suggests that the research methodology was rigorous and likely followed standard protocols for conducting clinical trials or observational studies.

2. **Sample Size and Population**: While specific details about the sample size and population demographics aren't provided, the fact that it's from a reputable journal indicates that the study had sufficient participants to draw meaningful conclusions.

3. **Methodology**: The use of a controlled experiment (a randomized controlled trial) would typically provide stronger evidence than observational studies. However, even observational studies with large sample sizes can still yield significant results if they meet certain criteria such as bl

AFTER (fine-tuned):
 Evidence type: Single small